# 01 — Data EDA (Week 1)

Exploratory analysis of `data/patient_records.csv`: distributions, missing values,
correlations and class balance. Run after `python src/data_pipeline.py`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

RAW = '../data/patient_records.csv'
CLEAN = '../data/patient_records_clean.csv'

df_raw = pd.read_csv(RAW)
df_clean = pd.read_csv(CLEAN)
print('Raw shape:', df_raw.shape, '| Clean shape:', df_clean.shape)
print('\nMissing values (raw):')
print(df_raw.isnull().sum())
print('\nMissing values (clean):', int(df_clean.isnull().sum().sum()))
df_raw.describe()

## Visualization 1 — Numeric feature distributions (raw vs cleaned)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['Age', 'Biomarker_Level', 'Vaccine_Dose_mcg']):
    ax.hist(df_raw[col].dropna(), bins=25, alpha=0.5, label='raw', color='gray')
    ax.hist(df_clean[col], bins=25, alpha=0.6, label='clean', color='#0F6E6E')
    ax.set_title(f'{col} distribution')
    ax.legend()
fig.tight_layout()
plt.show()

print('Biomarker_Level is right-skewed -> median imputation preserves the shape')
print('better than mean imputation would.')

## Visualization 2 — Correlation heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df_clean.corr(numeric_only=True), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation heatmap (cleaned data)')
fig.tight_layout()
plt.show()

## Visualization 3 — Class balance + risk factors

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

counts = df_clean['Reaction_Score'].value_counts().sort_index()
axes[0].bar(['Low risk (0)', 'High risk (1)'], counts.values, color=['#1F3A5F', '#B23A48'])
axes[0].set_title(f'Class balance ({counts[0]}/{counts[1]})')

sns.boxplot(data=df_clean, x='Reaction_Score', y='Biomarker_Level', ax=axes[1])
axes[1].set_title('Biomarker level by reaction score')

ct = pd.crosstab(df_clean['Prior_Reaction_History'], df_clean['Reaction_Score'], normalize='index')
ct.plot(kind='bar', stacked=True, ax=axes[2], color=['#1F3A5F', '#B23A48'])
axes[2].set_title('Reaction score share by prior history')
axes[2].legend(['Low', 'High'])
axes[2].tick_params(axis='x', rotation=0)

fig.tight_layout()
plt.show()

## Key takeaways

- The target is near-balanced (~51% / 49%), so accuracy is a meaningful metric
  and stratification is a safe choice.
- Higher `Biomarker_Level` and `Vaccine_Dose_mcg` lean toward `Reaction_Score = 1`,
  but the signal is noisy — believable, not suspiciously separable data.
- `Prior_Reaction_History` is missing in ~62% of rows (far more than the ~3.5%
  the PRD estimated), so mode imputation + a tree/linear model that can down-weight
  the feature is the pragmatic choice.